# Examples

In [1]:
%uv pip install -e .

Resolved 45 packages in 24ms                                         
   Building langmem @ file:///Users/xiangminli/Workspaces/github.com/sammyne/lan
      Built langmem @ file:///Users/xiangminli/Workspaces/github.com/sammyne/lan
Prepared 1 package in 178ms                                              
Uninstalled 1 package in 0.89ms
Installed 1 package in 1msom file:///Users/xiangminli/Worksp
 ~ langmem==0.0.30 (from file:///Users/xiangminli/Workspaces/github.com/sammyne/langmem)
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 安装 CPU 版 PyTorch，避免 sentence-transformers 自动安装 GPU 版 PyTorch
%uv pip install torch~=2.9 --index-url https://download.pytorch.org/whl/cpu
%uv pip install langchain-huggingface~=0.3 sentence-transformers~=5.2

Resolved 10 packages in 4.68s                                        
⠙ Preparing packages... (0/8)                                                   
⠙ Preparing packages... (0/8)-------------------     0 B/6.01 MiB            
⠙ Preparing packages... (0/8)-------------------     0 B/6.01 MiB            
mpmath               ------------------------------     0 B/523.63 KiB
⠙ Preparing packages... (0/8)-------------------     0 B/6.01 MiB            
mpmath               ------------------------------     0 B/523.63 KiB
⠙ Preparing packages... (0/8)------------------- 16.00 KiB/6.01 MiB          
mpmath               ------------------------------ 14.84 KiB/523.63 KiB
⠙ Preparing packages... (0/8)------------------- 16.00 KiB/6.01 MiB          
mpmath               ------------------------------ 14.84 KiB/523.63 KiB
⠙ Preparing packages... (0/8)------------------- 32.00 KiB/6.01 MiB          
mpmath               ------------------------------ 14.84 KiB/523.63 KiB
⠙ Preparing packages

## Hot Path Quickstart

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-mpnet-base-v2"
# model_name='BAAI/bge-small-en-v1.5'
embedding = HuggingFaceEmbeddings(model_name=model_name)

/Users/xiangminli/Workspaces/github.com/sammyne/langmem/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()


def new_openai_like(**kwargs) -> ChatOpenAI:
    return ChatOpenAI(
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.environ["OPENAI_API_BASE_URL"],
        model=os.environ["OPENAI_MODEL"],
        **kwargs,
    )

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langgraph.utils.config import get_store 
from langmem import (
    # Lets agent create, update, and delete memories 
    create_manage_memory_tool,
)


def prompt(state):
    """Prepare the messages for the LLM."""
    # Get store from configured contextvar; 
    store = get_store() # Same as that provided to `create_react_agent`
    memories = store.search(
        # Search within the same namespace as the one
        # we've configured for the agent
        ("memories",),
        query=state["messages"][-1].content,
    )
    system_msg = f"""You are a helpful assistant.

## Memories
<memories>
{memories}
</memories>
"""
    # print('----------')
    # print(system_msg)
    # print('----------')

    return [{"role": "system", "content": system_msg}, *state["messages"]]


store = InMemoryStore(
    index={ # Store extracted memories 
        "dims": 768,
        # "embed": "openai:text-embedding-3-small",
        "embed": embedding,
    }
) 
checkpointer = MemorySaver() # Checkpoint graph state 

model = new_openai_like()

agent = create_react_agent( 
    model,
    prompt=prompt,
    tools=[ # Add memory tools 
        # The agent can call "manage_memory" to
        # create, update, and delete memories by ID
        # Namespaces add scope to memories. To
        # scope memories per-user, do ("memories", "{user_id}"): 
        create_manage_memory_tool(namespace=("memories",)),
    ],
    # Our memories will be stored in this provided BaseStore instance
    store=store,
    # And the graph "state" will be checkpointed after each node
    # completes executing for tracking the chat history and durable execution
    checkpointer=checkpointer, 
)

In [4]:
config = {"configurable": {"thread_id": "thread-a"}}

# Use the agent. The agent hasn't saved any memories,
# so it doesn't know about us
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Know which display mode I prefer?"}
        ]
    },
    config=config,
)

# print(response["messages"][-1].content)
for v in response["messages"]:
    v.pretty_print()

----------
You are a helpful assistant.

## Memories
<memories>
[]
</memories>

----------
================================ Human Message =================================

Know which display mode I prefer?
================================== Ai Message ==================================

I don't currently have any stored information about your display mode preference. If you'd like, you can tell me your preferred display mode (e.g., dark mode, light mode, high contrast, etc.), and I can remember it for future conversations. Would you like to set a preference?


In [5]:
agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "dark. Remember that."}
        ]
    },
    # We will continue the conversation (thread-a) by using the config with
    # the same thread_id
    config=config,
)

# New thread = new conversation!
new_config = {"configurable": {"thread_id": "thread-b"}}
# The agent will only be able to recall
# whatever it explicitly saved using the manage_memories tool
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Hey there. Do you remember me? What are my preferences?"}]},
    config=new_config,
)

# print(response["messages"][-1].content)
for v in response["messages"]:
    v.pretty_print()

----------
You are a helpful assistant.

## Memories
<memories>
[]
</memories>

----------
----------
You are a helpful assistant.

## Memories
<memories>
[Item(namespace=['memories'], key='b7955923-9e6d-4575-92e8-e2c9646193c3', value={'content': 'User prefers dark display mode.'}, created_at='2025-12-31T14:05:14.798616+00:00', updated_at='2025-12-31T14:05:14.798620+00:00', score=0.04274036125014104)]
</memories>

----------
----------
You are a helpful assistant.

## Memories
<memories>
[Item(namespace=['memories'], key='b7955923-9e6d-4575-92e8-e2c9646193c3', value={'content': 'User prefers dark display mode.'}, created_at='2025-12-31T14:05:14.798616+00:00', updated_at='2025-12-31T14:05:14.798620+00:00', score=0.05718771720019353)]
</memories>

----------
================================ Human Message =================================

Hey there. Do you remember me? What are my preferences?
================================== Ai Message ==================================

Yes, I rememb